## Figure 2 - Compute transport plans

In [1]:
# env: pot
import numpy as np
import os
import ot
import pickle
import time

import meshio
import pyvista as pv

from scipy import sparse
from scipy.spatial import distance
# from sklearn.manifold import smacof
from sklearn.metrics import pairwise_distances

### Parameters

In [2]:
pe          = -0.5   # kPa 
alpha_lst   = [0.16] # kPa 
gravity_lst = [1]    # [0,1]

### Compute distance matrix and external cell areas

In [3]:
# Do at the same time supine and prone
acquisition_lst = []
acquisition_lst += ["ANTE_SUP2"]
acquisition_lst += ["ANTE_PRO1"]

volunteers_lst  = []
volunteers_lst += ["230526_02BT01"]
volunteers_lst += ["230704_02JY02"]
volunteers_lst += ["230706_02LH03"]
volunteers_lst += ["230728_02CL05"]
volunteers_lst += ["230804_02GA06"]
volunteers_lst += ["230929_02JD07"]
volunteers_lst += ["231013_02BF08"]
volunteers_lst += ["231020_02AR09"]
volunteers_lst += ["231024_02DL10"]
volunteers_lst += ["231027_02AS11"]
volunteers_lst += ["231110_02CC12"]
volunteers_lst += ["231117_02HC13"]
volunteers_lst += ["231124_02VG14"]
volunteers_lst += ["231205_02NS15"]
volunteers_lst += ["231219_02YF16"]
volunteers_lst += ["231222_02OC17"]
volunteers_lst += ["240105_02MD18"]
volunteers_lst += ["240109_02AL19"]
volunteers_lst += ["240119_02JN20"]
volunteers_lst += ["240202_02CH21"]
volunteers_lst += ["240206_02HA04"]
volunteers_lst += ["240524_02TA22"]
volunteers_lst += ["241011_02NB26"]
volunteers_lst += ["241018_02ZT27"]
volunteers_lst += ["241210_02JL29"]
volunteers_lst += ["241217_02JC30"]
volunteers_lst += ["250128_02RP32"]
volunteers_lst += ["250204_02MH35"]
volunteers_lst += ["250207_02SN36"]
volunteers_lst += ["250218_02TG23"]
volunteers_lst += ["250221_02VL38"]
volunteers_lst += ["250304_02WD39"]
volunteers_lst += ["250307_02MC41"]
volunteers_lst += ["250513_02WM46"]
volunteers_lst += ["250909_02LB54"]
volunteers_lst += ["250923_02FM56"]
volunteers_lst += ["250926_02AH57"]
volunteers_lst += ["251014_02RM58"]
volunteers_lst += ["251107_02SL59"]
volunteers_lst += ["251118_02JM60"]
volunteers_lst += ["251128_02LS24"]
volunteers_lst += ["251219_02GD67"]

# Repeated volunteers
# volunteers_lst += ["230718_02HA04"]
# volunteers_lst += ["240827_02TG23"]
# volunteers_lst += ["240903_02LS24"]

# Do separate by left and right lung
region_lst = []
# region_lst += ["LL"]
region_lst += ["RL"]

# exclude specific cases
exclude_lst  = []
exclude_lst += ["ANTE_PRO1-231222_02OC17"]
exclude_lst += ["ANTE_PRO2-231222_02OC17"]
exclude_lst += ["POST_PRO1-231222_02OC17"]
exclude_lst += ["ANTE_SUP2-250307_02MC41"]
exclude_lst += ["ANTE_PRO2-250307_02MC41"]
exclude_lst += ["ANTE_SUP2-251014_02RM58"]
exclude_lst += ["ANTE_PRO1-251014_02RM58"]
exclude_lst += ["ANTE_PRO2-251014_02RM58"]
exclude_lst += ["POST_PRO1-251014_02RM58"]
exclude_lst += ["POST_SUP1-251014_02RM58"]
exclude_lst += ["ANTE_PRO1-231013_02BF08-LL"] # mask not correct

In [4]:
C_list          = []
points_list     = []
areas_list      = []
info_list       = []

for acquisition in acquisition_lst:
    if acquisition in ["ANTE_SUP1", "ANTE_SUP2", "POST_SUP1"]:
        position = "supine"
    elif acquisition in ["ANTE_PRO1", "ANTE_PRO2", "POST_PRO1"]:
        position = "prone"
    else:
        raise ValueError(f"Unknown position for acquisition {acquisition}")
    
    for volunteer in volunteers_lst:
        if f"{acquisition}-{volunteer}" in exclude_lst:
            continue

        for region in region_lst:
            if f"{acquisition}-{volunteer}-{region}" in exclude_lst:
                continue
            
            print(f"Processing {acquisition} acquisition, {volunteer}, {region} region")

            for gravity_ in gravity_lst:
                if position == "supine":
                    gravity_ = gravity_ * 1
                elif position == "prone":
                    gravity_ = gravity_ * -1

                for alpha_   in alpha_lst:
                    resultsPath = f"./Results_{acquisition}/{volunteer}"

                    mesh_unloaded = pv.wrap(meshio.read(f"{resultsPath}/Pleural_pressure_estimation/mesh_unloaded_{region}_alpha{alpha_}_gravity{gravity_}_pe{pe}.xdmf"))

                    surf_mesh         = mesh_unloaded.extract_surface()
                    surf_cell_centers = surf_mesh.cell_centers() # Because the pleural pressure is constant in each cell face (of the boundary)

                    C = distance.cdist(surf_cell_centers.points, surf_cell_centers.points)
                    C_list.append(C)
                    points_list.append(surf_cell_centers.points)

                    ### External cells (faces) areas
                    areas = surf_mesh.compute_cell_sizes(length=False, area=True, volume=False)['Area']
                    areas_list.append(areas)

                    info = {"acquisition": acquisition, "ID": volunteer, "region": region, "alpha": alpha_, "gravity": gravity_, "pe": pe}
                    info_list.append(info)

Processing ANTE_SUP2 acquisition, 230526_02BT01, RL region
Processing ANTE_SUP2 acquisition, 230704_02JY02, RL region
Processing ANTE_SUP2 acquisition, 230706_02LH03, RL region
Processing ANTE_SUP2 acquisition, 230728_02CL05, RL region
Processing ANTE_SUP2 acquisition, 230804_02GA06, RL region
Processing ANTE_SUP2 acquisition, 230929_02JD07, RL region
Processing ANTE_SUP2 acquisition, 231013_02BF08, RL region
Processing ANTE_SUP2 acquisition, 231020_02AR09, RL region
Processing ANTE_SUP2 acquisition, 231024_02DL10, RL region
Processing ANTE_SUP2 acquisition, 231027_02AS11, RL region
Processing ANTE_SUP2 acquisition, 231110_02CC12, RL region
Processing ANTE_SUP2 acquisition, 231117_02HC13, RL region
Processing ANTE_SUP2 acquisition, 231124_02VG14, RL region
Processing ANTE_SUP2 acquisition, 231205_02NS15, RL region
Processing ANTE_SUP2 acquisition, 231219_02YF16, RL region
Processing ANTE_SUP2 acquisition, 231222_02OC17, RL region
Processing ANTE_SUP2 acquisition, 240105_02MD18, RL regi

In [5]:
Simulations_name = []
Simulations_name.append("alpha")
Simulations_name.append(alpha_lst)
Simulations_name.append("pe")
Simulations_name.append(pe)

flat_Simulations_name = [item for sublist in Simulations_name for item in (sublist if isinstance(sublist, list) else [sublist])]
Simulations_filename  = '_'.join([str(x) for x in flat_Simulations_name])

### Transport Plans

In [6]:
os.makedirs(f"TransportPlansPOT/{Simulations_filename}/", exist_ok=True)

In [7]:
# https://research.pasteur.fr/wp-content/uploads/2023/02/research_pasteur-computing-the-gromov-wasserstein-distance-between-two-surface-meshes-using-optimal-transport-algorithms-16-001311.pdf

patient_ref = 1 # Index (not patient ID) of reference mesh
C_target    = C_list[patient_ref]
q_target    = areas_list[patient_ref]/np.sum(areas_list[patient_ref])

C_target /= C_target.max()

print (f"Reference mesh: {info_list[patient_ref]['acquisition']}, {info_list[patient_ref]['ID']}, {info_list[patient_ref]['region']} region\n")

for ind, (C_source, info) in enumerate(zip(C_list, info_list)):
    if ind != patient_ref:
        t_ini = time.time()
        print(f"Computing Gromov-Wasserstein transport: {info['acquisition']}, {info['ID']}, {info['region']} region")

        # Compute Gromov-Wasserstein transport
        C_source /= C_source.max()
        p_source = areas_list[ind]/np.sum(areas_list[ind])

        # TODO Check why this works
        M  = pairwise_distances(points_list[ind], points_list[patient_ref])  # Euclidean cost matrix
        G0 = ot.emd(p_source, q_target, M, numItermax=200000)

        T_gw, log = ot.gromov.gromov_wasserstein(C1=C_source, C2=C_target, p=p_source, q=q_target, G0=G0, verbose=False, log=True)
        T_gw_sparse = sparse.csr_matrix(T_gw)

        with open(f"TransportPlansPOT/{Simulations_filename}/"
                  f"target_{info_list[patient_ref]['acquisition']}_{info_list[patient_ref]['ID'][-2:]}_{info_list[patient_ref]['region']}_"
                  f"source_{info['acquisition']}_{info['ID'][-2:]}_{info['region']}.pkl", "wb") as f:
            pickle.dump({"T":T_gw_sparse, "log": log, "info": info}, f)

        t_fin = time.time()
        print(f"Elapsed time: {time.strftime('%H:%M:%S', time.gmtime(t_fin - t_ini))}, GW distance: {log['gw_dist']:.2e}")

Reference mesh: ANTE_SUP2, 230704_02JY02, RL region

Computing Gromov-Wasserstein transport: ANTE_SUP2, 230526_02BT01, RL region
Elapsed time: 00:01:03, GW distance: 1.85e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 230706_02LH03, RL region
Elapsed time: 00:00:39, GW distance: 1.39e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 230728_02CL05, RL region
Elapsed time: 00:00:31, GW distance: 3.56e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 230804_02GA06, RL region
Elapsed time: 00:00:42, GW distance: 1.50e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 230929_02JD07, RL region
Elapsed time: 00:00:38, GW distance: 9.81e-04
Computing Gromov-Wasserstein transport: ANTE_SUP2, 231013_02BF08, RL region
Elapsed time: 00:01:22, GW distance: 1.40e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 231020_02AR09, RL region
Elapsed time: 00:00:26, GW distance: 2.42e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 231024_02DL10, RL region
Elapsed time: 00:

/Users/alvarez/miniconda3/envs/pot/lib/python3.13/site-packages/ot/lp/__init__.py:388: UserWarning: numItermax reached before optimality. Try to increase numItermax.
  result_code_string = check_result(result_code)


Elapsed time: 00:00:20, GW distance: 7.64e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 231124_02VG14, RL region
Elapsed time: 00:00:35, GW distance: 1.03e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 231205_02NS15, RL region
Elapsed time: 00:00:55, GW distance: 1.31e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 231219_02YF16, RL region
Elapsed time: 00:00:41, GW distance: 5.12e-04
Computing Gromov-Wasserstein transport: ANTE_SUP2, 231222_02OC17, RL region
Elapsed time: 00:01:06, GW distance: 2.80e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 240105_02MD18, RL region
Elapsed time: 00:00:56, GW distance: 1.44e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 240109_02AL19, RL region
Elapsed time: 00:00:43, GW distance: 1.00e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 240119_02JN20, RL region
Elapsed time: 00:00:29, GW distance: 1.47e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 240202_02CH21, RL region
Elapsed time: 00:00:56, 

/Users/alvarez/miniconda3/envs/pot/lib/python3.13/site-packages/ot/lp/__init__.py:388: UserWarning: numItermax reached before optimality. Try to increase numItermax.
  result_code_string = check_result(result_code)


Elapsed time: 00:00:25, GW distance: 6.02e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 241217_02JC30, RL region
Elapsed time: 00:00:43, GW distance: 3.16e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 250128_02RP32, RL region
Elapsed time: 00:00:29, GW distance: 1.92e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 250204_02MH35, RL region
Elapsed time: 00:00:57, GW distance: 4.10e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 250207_02SN36, RL region
Elapsed time: 00:01:06, GW distance: 5.17e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 250218_02TG23, RL region
Elapsed time: 00:01:34, GW distance: 6.64e-03
Computing Gromov-Wasserstein transport: ANTE_SUP2, 250221_02VL38, RL region
Elapsed time: 00:00:37, GW distance: 7.32e-04
Computing Gromov-Wasserstein transport: ANTE_SUP2, 250304_02WD39, RL region
Elapsed time: 00:00:41, GW distance: 8.60e-04
Computing Gromov-Wasserstein transport: ANTE_SUP2, 250513_02WM46, RL region
Elapsed time: 00:00:45, 

...

Elapsed time: 00:00:43, GW distance: 8.35e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 231124_02VG14, RL region
Elapsed time: 00:00:55, GW distance: 3.07e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 231205_02NS15, RL region
Elapsed time: 00:01:17, GW distance: 5.00e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 231219_02YF16, RL region
Elapsed time: 00:01:02, GW distance: 3.21e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 240105_02MD18, RL region
Elapsed time: 00:00:58, GW distance: 1.94e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 240109_02AL19, RL region
Elapsed time: 00:00:42, GW distance: 1.44e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 240119_02JN20, RL region
Elapsed time: 00:00:40, GW distance: 1.45e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 240202_02CH21, RL region
Elapsed time: 00:01:07, GW distance: 1.68e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 240206_02HA04, RL region
Elapsed time: 00:00:54, 

/Users/alvarez/miniconda3/envs/pot/lib/python3.13/site-packages/ot/lp/__init__.py:388: UserWarning: numItermax reached before optimality. Try to increase numItermax.
  result_code_string = check_result(result_code)


Elapsed time: 00:00:26, GW distance: 6.85e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 241217_02JC30, RL region
Elapsed time: 00:00:52, GW distance: 6.53e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 250128_02RP32, RL region
Elapsed time: 00:01:18, GW distance: 3.20e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 250204_02MH35, RL region
Elapsed time: 00:00:53, GW distance: 6.36e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 250207_02SN36, RL region
Elapsed time: 00:01:26, GW distance: 5.15e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 250218_02TG23, RL region
Elapsed time: 00:02:24, GW distance: 8.23e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 250221_02VL38, RL region
Elapsed time: 00:00:50, GW distance: 3.12e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 250304_02WD39, RL region
Elapsed time: 00:00:34, GW distance: 1.39e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 250307_02MC41, RL region
Elapsed time: 00:00:53, 

/Users/alvarez/miniconda3/envs/pot/lib/python3.13/site-packages/ot/lp/__init__.py:388: UserWarning: numItermax reached before optimality. Try to increase numItermax.
  result_code_string = check_result(result_code)


Elapsed time: 00:03:18, GW distance: 4.74e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 250923_02FM56, RL region
Elapsed time: 00:00:29, GW distance: 1.30e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 250926_02AH57, RL region
Elapsed time: 00:00:37, GW distance: 1.73e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 251107_02SL59, RL region
Elapsed time: 00:00:44, GW distance: 1.12e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 251118_02JM60, RL region
Elapsed time: 00:00:42, GW distance: 3.37e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 251128_02LS24, RL region
Elapsed time: 00:01:25, GW distance: 3.34e-03
Computing Gromov-Wasserstein transport: ANTE_PRO1, 251219_02GD67, RL region
Elapsed time: 00:05:53, GW distance: 1.79e-03
